In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torch.utils.data import ConcatDataset, DataLoader

import sys
import os

current_dir = os.path.dirname(os.path.abspath(__file__))
sys.path.append(current_dir)

from utils.EEDDataLoader import EEGDataset
from utils.train_eval import eval, train
from models.ConvParallelEEG1DModel import ConvParallelEEG1DModel

files_list_csv = "./dataset/meta_data.csv"
train_data_csv = "./dataset/train.csv"
val_data_csv = "./dataset/eval.csv"
test_data_csv = "./dataset/test.csv"
prediction_saving_csv_train = "./dataset/predictions/predictions_train_conv_crops.csv"
prediction_saving_csv_eval = "./dataset/predictions/predictions_eval_conv_crops.csv"
prediction_saving_csv_test = "./dataset/predictions/predictions_test_conv_crops.csv"
model_saving_name = "./dataset/model/weights/conv_tt_0_original.pt"

# epochs = 50
epochs = 5
batch_size = 64
hidden_size = 20
output_size = 3  # Binary classification
number_of_eeg_channels = 1
eval_batch = 1
threshold = 0.495
active_branches = [0,1,2]

In [12]:
train_dataset = EEGDataset(train_data_csv, number_of_eeg_channels=number_of_eeg_channels)
val_dataset = EEGDataset(val_data_csv, number_of_eeg_channels=number_of_eeg_channels)
test_dataset = EEGDataset(test_data_csv, number_of_eeg_channels=number_of_eeg_channels)
combined_dataset = ConcatDataset([train_dataset, val_dataset])

# Split the data into training and validation sets
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)
train_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)

val_loader = DataLoader(val_dataset, batch_size=eval_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=eval_batch, shuffle=False)

In [13]:
input = train_dataset.__getitem__(0)
input_x1 = input[1].shape
input_x2 = input[2].shape
input_x3 = input[3].shape
criterion = nn.CrossEntropyLoss()

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

print(input_x1,input_x2,input_x3)
input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

model = ConvParallelEEG1DModel(input_channels_list=input_channels_list, output_size=3)

main_output, aux_outputs = model([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output.shape) 

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

torch.Size([1, 1, 400]) torch.Size([1, 33, 14]) torch.Size([1, 30, 400])
Output shape: torch.Size([1, 3])


/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


ConvParallelEEG1DModel(
  (branches): ModuleList(
    (0): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(in_features=32, out_features=3, bias=True)
    )
    (1): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(33, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(

In [14]:
prev_eval_loss = 0
prev_eval_acc = 0


In [21]:
model = model.to(device)

prev_eval_loss, prev_eval_acc = eval(model, val_loader, loss_fn, 1,False,'', active_branches, device)
print("Loss:",prev_eval_loss, "Accuracy",prev_eval_acc)

  0%|          | 0/43343 [00:00<?, ?it/s]

Correct classified: 14956.0 / 43343
Class-wise correct: Class 0 = 0.0, Class 1 = 14956.0, Class 2 = 0.0
Precision: 11.50  Recall: 33.33  F1: 17.10
Confusion Matrix:
 [[    0 22538     0]
 [    0 14956     0]
 [    0  5849     0]]
Loss: 1.0925154486214743 Accuracy 0.345061486283829


/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [15]:
result = {"epoch":[],"train_loss":[],"train_acc":[],"eval_loss":[],"eval_acc":[],"test_acc":[],"test_loss":[]}
saved_epoch = 0

In [16]:
for epoch in range(epochs):  # loop over the dataset multiple times
    train_loss, train_acc = train(model, train_loader,optimizer, loss_fn, active_branches, device)
    print(f"Train Loss after epoch {epoch+2}: {train_loss} Train ACC after epoch {epoch}: {train_acc}")
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    print(f"VAL Loss after epoch {epoch}: {eval_loss} VAL ACC after epoch {epoch}: {eval_acc}")
    # test_loss, test_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    test_loss = eval_loss
    test_acc = eval_acc
    print(f"Test Loss after epoch {epoch}: {test_loss} Test Acc after epoch {epoch}: {test_acc}")

    scheduler.step(eval_loss)
    
    result['epoch'].append(epoch)
    
    result['train_loss'].append(train_loss)
    result['train_acc'].append(train_acc)
    result['eval_loss'].append(eval_loss)
    result['eval_acc'].append(eval_acc)
    result['test_loss'].append(test_loss)
    result['test_acc'].append(test_acc)

    if prev_eval_acc < eval_acc:
      prev_eval_acc = eval_acc
      saved_epoch = epoch
      torch.save(model.state_dict(), model_saving_name)
      print("WEIGHTS SAVED")

Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x76b37a3d3310>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
Training: 100%|██████████| 4531/4531 [10:47<00:00,  7.00it/s]


Correct Classified: 206262.0 / 289972
Class-wise correct: Class 0 = 122902.0, Class 1 = 29878.0, Class 2 = 53482.0
Train Loss after epoch 2: 1.626957754166914 Train ACC after epoch 0: 0.7113169547404578


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x76b37a3d3310>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 74774/74774 [04:19<00:00, 288.55it/s]


Correct classified: 56704.0 / 74774
Class-wise correct: Class 0 = 31644.0, Class 1 = 9173.0, Class 2 = 15887.0
Precision: 71.62  Recall: 73.21  F1: 71.99
Confusion Matrix:
 [[31644  1737  3182]
 [ 1224  9173  2775]
 [ 3186  5966 15887]]
VAL Loss after epoch 0: 0.7100682916426042 VAL ACC after epoch 0: 0.758338459892476
Test Loss after epoch 0: 0.7100682916426042 Test Acc after epoch 0: 0.758338459892476
WEIGHTS SAVED


Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x76b37a3d3310>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Training: 100%|██████████| 4531/4531 [10:54<00:00,  6.92it/s]


...

Correct classified: 56933.0 / 74774
Class-wise correct: Class 0 = 32430.0, Class 1 = 8025.0, Class 2 = 16478.0
Precision: 70.97  Recall: 71.81  F1: 71.17
Confusion Matrix:
 [[32430  1669  2464]
 [ 1585  8025  3562]
 [ 2415  6146 16478]]
VAL Loss after epoch 3: 0.685725551665644 VAL ACC after epoch 3: 0.7614010217455265
Test Loss after epoch 3: 0.685725551665644 Test Acc after epoch 3: 0.7614010217455265


Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x76b37a3d3310>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Training: 100%|██████████| 4531/4531 [10:49<00:00,  6.98it/s]


Correct Classified: 228645.0 / 289972
Class-wise correct: Class 0 = 123821.0, Class 1 = 39140.0, Class 2 = 65684.0
Train Loss after epoch 6: 1.4049895638215624 Train ACC after epoch 4: 0.7885071662091512


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x76b37a3d3310>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 74774/74774 [04:14<00:00, 293.46it/s]


Correct classified: 56999.0 / 74774
Class-wise correct: Class 0 = 32547.0, Class 1 = 7553.0, Class 2 = 16899.0
Precision: 70.76  Recall: 71.28  F1: 70.91
Confusion Matrix:
 [[32547  1457  2559]
 [ 1994  7553  3625]
 [ 2153  5987 16899]]
VAL Loss after epoch 4: 0.6743215884579205 VAL ACC after epoch 4: 0.7622836814935673
Test Loss after epoch 4: 0.6743215884579205 Test Acc after epoch 4: 0.7622836814935673
WEIGHTS SAVED


In [17]:
print("Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  ")
for epoch in range(epochs):
    if(saved_epoch == epoch):
        print("Saved Weights")
    print(result['epoch'][epoch],"|",result['train_loss'][epoch],"|",result['train_acc'][epoch],"|",result['eval_loss'][epoch],"|",result['eval_acc'][epoch],"|",result['test_loss'][epoch],"|",result['test_acc'][epoch])

Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  
0 | 1.626957754166914 | 0.7113169547404578 | 0.7100682916426042 | 0.758338459892476 | 0.7100682916426042 | 0.758338459892476
1 | 1.4702444528801977 | 0.767111997020402 | 0.6941901408828398 | 0.7621766924331987 | 0.6941901408828398 | 0.7621766924331987
2 | 1.4361430775169453 | 0.7780509842329604 | 0.6817300254837012 | 0.7592746141707011 | 0.6817300254837012 | 0.7592746141707011
3 | 1.416674174448955 | 0.7844550508324942 | 0.685725551665644 | 0.7614010217455265 | 0.685725551665644 | 0.7614010217455265
Saved Weights
4 | 1.4049895638215624 | 0.7885071662091512 | 0.6743215884579205 | 0.7622836814935673 | 0.6743215884579205 | 0.7622836814935673


In [19]:
model_state_dict = torch.load(model_saving_name)
# Load the state_dict into the model
model.load_state_dict(model_state_dict)
test_loss, test_acc = eval(model, test_loader, loss_fn, eval_batch,True,prediction_saving_csv_eval, active_branches, device)
print(f"Test Loss: {test_loss} Test ACC: {test_acc}")

# eval_loss, eval_acc = eval(model, train_loader, loss_fn, eval_batch,True,prediction_saving_csv_train)
# print(f"Train Loss: {eval_loss} Train ACC: {eval_acc}")

/tmp/ipykernel_938217/1453154948.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_state_dict = torch.load(model_saving_name)
Eval:   0%|          | 0/74774 [00:00<?

KeyboardInterrupt: 